In [1]:
import sqlite3
import time
import pandas as pd
import numpy as np

# 1. Crear conexión a una base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Crear la tabla 'ventas'
cursor.execute('''
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT,
    cliente_id INTEGER,
    categoria TEXT,
    monto REAL,
    observacion TEXT
)
''')

# 3. Insertar 100,000 registros simulados
np.random.seed(42)
n_registros = 100000

fechas = pd.date_range(start='2023-01-01', end='2025-12-31', periods=n_registros)
fechas_str = fechas.strftime('%Y-%m-%d').tolist()

datos = [
    (
        fechas_str[i],
        int(np.random.randint(1000, 5000)),
        np.random.choice(['Electronica', 'Hogar', 'Ropa', 'Alimentos']),
        float(np.round(np.random.uniform(10.0, 500.0), 2)),
        'Transaccion realizada con exito en sucursal'
    )
    for i in range(n_registros)
]

cursor.executemany('''
INSERT INTO ventas (fecha, cliente_id, categoria, monto, observacion)
VALUES (?, ?, ?, ?, ?)
''', datos)

conn.commit()
print(f"¡Base de datos inicializada con éxito! {n_registros} registros insertados en la tabla 'ventas'.")

# Medir el tiempo de ejecución de la consulta LENTA
inicio = time.time()

# Consulta con anti-patrones: SELECT * y función en el WHERE
query_lenta = """
    SELECT *
    FROM ventas
    WHERE SUBSTR(fecha, 1, 4) = '2024' AND categoria = 'Electronica'
"""

cursor.execute(query_lenta)
resultado_lento = cursor.fetchall()

fin = time.time()
tiempo_lento = fin - inicio

print(f"--- ANÁLISIS DE LA CONSULTA LENTA ---")
print(f"Filas obtenidas: {len(resultado_lento)}")
print(f"Tiempo de ejecución: {tiempo_lento:.6f} segundos")

# 1. Crear un índice en las columnas de filtrado para acelerar la búsqueda
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ventas_fecha_cat ON ventas(fecha, categoria);")
conn.commit()

# Medir el tiempo de ejecución de la consulta OPTIMIZADA
inicio = time.time()

# Consulta optimizada: Columnas específicas, rango directo y soporte de índice
query_optimizada = """
    SELECT id_venta, fecha, cliente_id, monto
    FROM ventas
    WHERE fecha >= '2024-01-01' AND fecha < '2025-01-01'
      AND categoria = 'Electronica'
"""

cursor.execute(query_optimizada)
resultado_optimizado = cursor.fetchall()

fin = time.time()
tiempo_optimizado = fin - inicio

print(f"--- ANÁLISIS DE LA CONSULTA OPTIMIZADA ---")
print(f"Filas obtenidas: {len(resultado_optimizado)}")
print(f"Tiempo de ejecución: {tiempo_optimizado:.6f} segundos")
print(f"Mejora de velocidad: {tiempo_lento / tiempo_optimizado:.2f}x más rápido.")



¡Base de datos inicializada con éxito! 100000 registros insertados en la tabla 'ventas'.
--- ANÁLISIS DE LA CONSULTA LENTA ---
Filas obtenidas: 8382
Tiempo de ejecución: 0.032136 segundos
--- ANÁLISIS DE LA CONSULTA OPTIMIZADA ---
Filas obtenidas: 8382
Tiempo de ejecución: 0.020067 segundos
Mejora de velocidad: 1.60x más rápido.
